# DreamPrice Quick Start

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SharathSPhD/dreamprice/blob/master/notebooks/quickstart.ipynb)

This notebook walks through a minimal end-to-end example: create a pricing
environment, run a rollout with random and hold-price policies, and visualize
the results.

The environment uses a log-linear demand fallback when no trained world model
is provided, making this notebook runnable anywhere — CPU, Colab, or DGX Spark.

In [ ]:
# Install DreamPrice (skip if already installed)
import subprocess, sys
try:
    import retail_world_model
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           'torch', '--index-url', 'https://download.pytorch.org/whl/cpu'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '..'])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from retail_world_model.envs.grocery import GroceryPricingEnv

print('Imports OK')

In [ ]:
# Create a pricing environment with 3 SKUs, 13-week episodes
# world_model=None activates the built-in log-linear demand model
env = GroceryPricingEnv(
    world_model=None,
    store_features=np.zeros(8),
    initial_obs=np.zeros(32),
    cost_vector=np.array([1.80, 0.90, 2.10]),
    n_skus=3,
    H=13,
)

obs, info = env.reset(seed=42)
print(f'Observation shape: {obs.shape}')
print(f'Action space: {env.action_space}')

In [ ]:
# Run a 13-step rollout with random actions
prices_list = [info.get('prices', np.ones(3))]
rewards = []
obs, info = env.reset(seed=42)

for step in range(13):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    prices_list.append(info.get('prices', np.ones(3)))
    rewards.append(reward)
    if terminated or truncated:
        break

prices = np.array(prices_list)
rewards = np.array(rewards)
print(f'Episode length: {len(rewards)} steps')
print(f'Total reward: {rewards.sum():.2f}')

In [ ]:
# Plot price trajectories for each SKU
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for sku in range(prices.shape[1]):
    axes[0].plot(prices[:, sku], label=f'SKU {sku}')
axes[0].set_xlabel('Week')
axes[0].set_ylabel('Price ($)')
axes[0].set_title('Price Trajectories')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].bar(range(len(rewards)), rewards, color='steelblue', alpha=0.7)
axes[1].set_xlabel('Week')
axes[1].set_ylabel('Reward')
axes[1].set_title('Weekly Rewards')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Compare random vs hold-price policies
def run_policy(env, policy='random', n_episodes=10, seed_start=0):
    """Run episodes and return per-episode total rewards."""
    returns = []
    for ep in range(n_episodes):
        obs, info = env.reset(seed=seed_start + ep)
        total_reward = 0.0
        done = False
        while not done:
            if policy == 'random':
                action = env.action_space.sample()
            elif policy == 'hold':
                action = np.full(env.n_skus, 10, dtype=int)  # 1.00x = no change
            else:
                action = env.action_space.sample()
            obs, reward, terminated, truncated, info = env.step(action)
            total_reward += reward
            done = terminated or truncated
        returns.append(total_reward)
    return np.array(returns)

random_returns = run_policy(env, policy='random', n_episodes=20)
hold_returns = run_policy(env, policy='hold', n_episodes=20)

print(f'Random policy: mean={random_returns.mean():.2f} +/- {random_returns.std():.2f}')
print(f'Hold policy:   mean={hold_returns.mean():.2f} +/- {hold_returns.std():.2f}')

In [ ]:
# Episode reward breakdown
print('Episode Summary')
print('=' * 40)
print(f'Total reward:   {rewards.sum():>10.2f}')
print(f'Mean reward:    {rewards.mean():>10.2f}')
print(f'Std reward:     {rewards.std():>10.2f}')
print(f'Min reward:     {rewards.min():>10.2f}')
print(f'Max reward:     {rewards.max():>10.2f}')
print(f'Final prices:   {prices[-1]}')